# Multi-Task Neural Network Pipeline
```
Input (2242 features)
    ↓  BatchNorm
    ↓  Linear 2242→1024  BatchNorm  ReLU  Dropout(0.3)  
    ↓  Linear 1024→512   BatchNorm  ReLU  Dropout(0.3)  
    ↓  Linear  512→256   BatchNorm  ReLU  Dropout(0.3)  
    ↓ ↓ ↓ ... ↓   (12 task heads)
    
    Linear 256→64  ReLU  Dropout(0.15)  Linear 64→1  Sigmoid
```


##  Importing Librabries

In [1]:
# ── Standard libraries ───────────────────────────────────────────
import os
import sys
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# ── RDKit — chemistry library ────────────────────────────────────

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, MACCSkeys, QED
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.rdMolDescriptors import (
    CalcTPSA, CalcNumHBD, CalcNumHBA,
    CalcNumRotatableBonds, CalcNumAromaticRings,
    CalcFractionCSP3, CalcNumRings,
    CalcNumSpiroAtoms, CalcNumAtomStereoCenters,
)

# ── PyTorch — neural network framework ──────────────────────────
# torch      = core tensor operations (like numpy but GPU-capable)
# nn         = building blocks: layers, activations, loss functions
# optim      = optimizers (Adam, SGD etc.)
# DataLoader = feeds data in batches during training
# Dataset    = wraps our arrays into PyTorch-compatible format
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# ── Imbalance handling ───────────────────────────────────────────
# SMOTE = Synthetic Minority Oversampling Technique
# Creates new synthetic toxic compounds by interpolating between
# existing ones — better than just reweighting (scale_pos_weight)
from imblearn.over_sampling import SMOTE

# ── Sklearn utilities ────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
from sklearn.preprocessing import StandardScaler

# ── Output folders ────────────────────────────────────────────────
os.makedirs('mt_plots',   exist_ok=True)
os.makedirs('mt_models',  exist_ok=True)
os.makedirs('mt_results', exist_ok=True)

# ── Device: GPU if available, else CPU ───────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version : {torch.__version__}')
print(f'Using device    : {DEVICE}')
print('All imports OK')


PyTorch version : 2.1.0+cpu
Using device    : cpu
All imports OK


##  Fixation of Parameters

In [2]:
# ── The 12 toxicity endpoints we are predicting ──────────────────
TOX21_ENDPOINTS = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase',
    'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
    'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'
]

# ── Neural network hyperparameters ───────────────────────────────


BATCH_SIZE    = 128    
                       

EPOCHS        = 200    # Maximum training loops through full dataset
                       # Early stopping will usually stop before this

LEARNING_RATE = 1e-4   # Step size for weight updates (0.001)
                       

DROPOUT_RATE  = 0.1    # Fraction of neurons randomly silenced per batch
                       

PATIENCE      = 25     
                       # for this many consecutive epochs

RANDOM_STATE  = 42     # Seed — ensures identical results every run

print('Parameters defined')
print(f'Endpoints  : {len(TOX21_ENDPOINTS)}')
print(f'Batch size : {BATCH_SIZE}')
print(f'Max epochs : {EPOCHS}  (early stopping patience={PATIENCE})')
print(f'Device     : {DEVICE}')


Parameters defined
Endpoints  : 12
Batch size : 128
Max epochs : 200  (early stopping patience=25)
Device     : cpu


## Loading Data & Parsing SMILES

In [3]:
# ── Load tox21.csv ───────────────────────────────────────────────
df = pd.read_csv('tox21.csv')
print(f'Raw data shape: {df.shape}')

# ── Confirm which endpoints exist in this file ───────────────────
toxicity_labels = [e for e in TOX21_ENDPOINTS if e in df.columns]
print(f'Endpoints found: {len(toxicity_labels)}/12')

# ── Parse SMILES → RDKit molecule objects ────────────────────────
# SMILES (e.g. "CCOc1ccc2nc...") is just a text string.
# RDKit converts it into a real molecule object we can compute features from.
# Invalid or malformed SMILES are silently dropped.

mols, valid_rows, invalid_rows = [], [], []
for i, smi in enumerate(df['smiles']):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is not None:
        mols.append(mol)
        valid_rows.append(i)
    else:
        invalid_rows.append(i)

df = df.iloc[valid_rows].reset_index(drop=True)

print(f'Valid molecules  : {len(mols):,}')
print(f'Dropped (invalid): {len(invalid_rows)} (Al valence errors, harmless)')
print(f'Working dataset  : {len(df):,} compounds')


Raw data shape: (7831, 14)
Endpoints found: 12/12


[21:51:56] WARNING: not removing hydrogen atom without neighbors


Valid molecules  : 7,831
Dropped (invalid): 0 (Al valence errors, harmless)
Working dataset  : 7,831 compounds


## Feature Engineering (All 5 Layers)

We built 2242 features from SMILES strings using RDKit:

- Layer 1: Morgan fingerprints (2048 bits) — molecular barcode
- Layer 2: MACCS keys (167 bits) — interpretable structural keys
- Layer 3: RDKit descriptors (10 values) — physicochemical properties
- Layer 4: Engineered features (10 values) — Lipinski flags + interactions
- Layer 5: ZINC properties (7 values) — drug-likeness filters


In [4]:
print('Building features... (1-2 minutes)')

# LAYER 1 — Morgan Fingerprints (2048 bits)

# Encodes the chemical environment around every atom up to
# radius=2 bonds away. Result: 2048-bit binary vector per molecule.


morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
morgan_arr = np.array(
    [list(morgan_gen.GetFingerprint(mol)) for mol in mols],
    dtype=np.float32
)
print(f'Layer 1 Morgan      : {morgan_arr.shape}')

# LAYER 2 — MACCS Structural Keys (167 bits)

# 167 pre-defined structural questions about the molecule.
# e.g. bit 162 = "does it have an aromatic ring?"
#      bit 125 = "does it have a carbonyl C=O group?"

maccs_arr = np.array(
    [list(MACCSkeys.GenMACCSKeys(mol)) for mol in mols],
    dtype=np.float32
)
print(f'Layer 2 MACCS       : {maccs_arr.shape}')


# LAYER 3 — RDKit Physicochemical Descriptors (10 values)
# Continuous numerical properties — same as Lipinski's Rule of 5.

def compute_descriptors(mol):
    try:
        return [
            Descriptors.MolWt(mol),           # molecular weight (g/mol)
            Descriptors.MolLogP(mol),          # fat solubility
            QED.qed(mol),                      # drug-likeness score 0-1
            CalcTPSA(mol),                     # polar surface area
            CalcNumHBD(mol),                   # hydrogen bond donors
            CalcNumHBA(mol),                   # hydrogen bond acceptors
            CalcNumRotatableBonds(mol),         # molecular flexibility
            CalcNumAromaticRings(mol),          # number of aromatic rings
            CalcFractionCSP3(mol),             # fraction of sp3 carbons
            mol.GetNumHeavyAtoms(),            # molecular size
        ]
    except Exception:
        return [np.nan] * 10

desc_arr = np.array([compute_descriptors(mol) for mol in mols], dtype=np.float32)
print(f'Layer 3 Descriptors : {desc_arr.shape}')


# LAYER 4 — Engineered Features (10 values)
# Hand-crafted features combining domain knowledge:


MW   = desc_arr[:, 0]
logP = desc_arr[:, 1]
TPSA = desc_arr[:, 3]
HBD  = desc_arr[:, 4]
HBA  = desc_arr[:, 5]

eng_arr = np.column_stack([
    (MW   < 500).astype(float),          # passes Lipinski MW rule
    (logP < 5  ).astype(float),          # passes Lipinski logP rule
    (HBD  <= 5 ).astype(float),          # passes Lipinski HBD rule
    (HBA  <= 10).astype(float),          # passes Lipinski HBA rule
    ((MW<500)&(logP<5)&(HBD<=5)&(HBA<=10)).astype(float),   # passes all 4
    (4 - ((MW<500).astype(int) + (logP<5).astype(int) +
          (HBD<=5).astype(int) + (HBA<=10).astype(int))).astype(float),
    logP * MW,                           # logP × MW interaction term
    TPSA / (MW + 1),                     # normalised TPSA
    HBD + HBA,                           # total H-bond capacity
    np.log1p(MW),                        # log(MW) — corrects right skew
]).astype(np.float32)
print(f'Layer 4 Engineered  : {eng_arr.shape}')


# LAYER 5 — ZINC Drug-likeness Properties (7 values)

# ZINC is a database of drug-like compounds. These features encode
# whether a molecule fits standard drug-development categories.

def compute_zinc(mol):
    try:
        mw   = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        qed  = QED.qed(mol)
        sas  = min(max(
            1.0 + 0.25*CalcNumRings(mol)
                + 0.04*mol.GetNumHeavyAtoms()
                + 0.5*CalcNumAtomStereoCenters(mol)
                + 0.8*CalcNumSpiroAtoms(mol),
            1.0), 10.0)
        return [
            logp,                                  # logP (ZINC scale)
            qed,                                   # drug-likeness 0-1
            sas,                                   # synthetic accessibility
            round(10.0 - sas, 3),                  # inverted SAS (higher=easier)
            int(150 < mw < 500 and -0.4 < logp < 5.6),  # drug-like?
            int(mw < 300 and logp < 3),            # fragment-like?
            int(250 < mw < 350 and -1 < logp < 3.5),    # lead-like?
        ]
    except Exception:
        return [np.nan] * 7

zinc_arr = np.array([compute_zinc(mol) for mol in mols], dtype=np.float32)
print(f'Layer 5 ZINC        : {zinc_arr.shape}')


# COMBINE all 5 layers into one feature matrix
# [morgan(2048) | maccs(167) | desc(10) | eng(10) | zinc(7)] = 2242 columns

X_raw = np.hstack([morgan_arr, maccs_arr, desc_arr, eng_arr, zinc_arr])
print(f'\nCombined feature matrix : {X_raw.shape}')
print(f'NaN count before imputation: {np.isnan(X_raw).sum()}')

#  Imputing feature NaNs with column median 

col_medians = np.nanmedian(X_raw, axis=0)
nan_mask    = np.isnan(X_raw)
X_raw[nan_mask] = np.take(col_medians, np.where(nan_mask)[1])
print(f'NaN count after imputation : {np.isnan(X_raw).sum()} (must be 0)')

# Build label matrix 

Y_raw = df[toxicity_labels].values.astype(np.float32)

print(f'\nFeature matrix : {X_raw.shape}')
print(f'Label matrix   : {Y_raw.shape}')
print(f'Label NaN count: {np.isnan(Y_raw).sum():,} (expected — by design)')


Building features... (1-2 minutes)
Layer 1 Morgan      : (7831, 2048)
Layer 2 MACCS       : (7831, 167)


[21:53:37] WARNING: not removing hydrogen atom without neighbors


Layer 3 Descriptors : (7831, 10)
Layer 4 Engineered  : (7831, 10)


[21:54:17] WARNING: not removing hydrogen atom without neighbors


Layer 5 ZINC        : (7831, 7)

Combined feature matrix : (7831, 2242)
NaN count before imputation: 0
NaN count after imputation : 0 (must be 0)

Feature matrix : (7831, 2242)
Label matrix   : (7831, 12)
Label NaN count: 16,026 (expected — by design)


##  Normalizing Features



In [ ]:
# Train/validation split BEFORE scaling 


strat_labels = np.where(np.isnan(Y_raw[:, 0]), -1, Y_raw[:, 0]).astype(int)

train_idx, val_idx = train_test_split(
    np.arange(len(X_raw)),
    test_size    = 0.15,          # 85% train, 15% validation
    random_state = RANDOM_STATE,
    stratify     = strat_labels   # same toxic% in both splits
)

X_train_raw, X_val_raw = X_raw[train_idx], X_raw[val_idx]
Y_train,     Y_val     = Y_raw[train_idx], Y_raw[val_idx]

print(f'Train set : {X_train_raw.shape[0]:,} compounds')
print(f'Val set   : {X_val_raw.shape[0]:,} compounds')

# Fitting scaler on training data ONLY 

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)  
X_val   = scaler.transform(X_val_raw).astype(np.float32)       

with open('mt_models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f'\nAfter scaling — Train mean (should be ~0): {X_train.mean():.4f}')
print(f'After scaling — Train std  (should be ~1): {X_train.std():.4f}')
print('Scaler saved → mt_models/scaler.pkl')


Train set : 6,656 compounds
Val set   : 1,175 compounds

After scaling — Train mean (should be ~0): 0.0000
After scaling — Train std  (should be ~1): 0.9991
Scaler saved → mt_models/scaler.pkl


##  SMOTE: Handling Class Imbalance



In [ ]:
# Apply SMOTE per endpoint 


smote_data    = {}   # smote_data[ep] = (X_balanced, y_balanced)
class_weights = {}   # residual imbalance after SMOTE → used in loss

print('Applying SMOTE per endpoint...')
print(f'{"Endpoint":<22} {"Before":>10} {"After":>10} {"Toxic before":>14} {"Toxic after":>12}')
print('-' * 72)

for i, ep in enumerate(toxicity_labels):

    
    ep_mask = ~np.isnan(Y_train[:, i])    
    X_ep    = X_train[ep_mask]            
    y_ep    = Y_train[ep_mask, i].astype(int)  

    n_toxic  = int((y_ep == 1).sum())
    n_safe   = int((y_ep == 0).sum())
    n_before = len(y_ep)

    
    if n_toxic >= 10:
        smote = SMOTE(
            
            sampling_strategy = min(0.5, n_toxic / n_safe + 0.2),
            random_state      = RANDOM_STATE,
            k_neighbors       = min(5, n_toxic - 1),
        )
        try:
            X_bal, y_bal = smote.fit_resample(X_ep, y_ep)
        except Exception:
            X_bal, y_bal = X_ep, y_ep   
    else:
        X_bal, y_bal = X_ep, y_ep       

    smote_data[ep] = (X_bal, y_bal)

.
    n_t_after = int((y_bal == 1).sum())
    n_s_after = int((y_bal == 0).sum())
    class_weights[ep] = n_s_after / max(n_t_after, 1)

    print(f'{ep:<22} {n_before:>10,} {len(y_bal):>10,} '
          f'{n_toxic:>14,} {n_t_after:>12,}')

print('\nSMOTE complete.')
print('Residual class weights passed to loss function:')
for ep, w in class_weights.items():
    print(f'  {ep:<22}: {w:.2f}x')


Applying SMOTE per endpoint...
Endpoint                   Before      After   Toxic before  Toxic after
------------------------------------------------------------------------
NR-AR                       6,175      7,357            263        1,445
NR-AR-LBD                   5,751      6,861            201        1,311
NR-AhR                      5,588      6,576            648        1,636
NR-Aromatase                4,965      5,907            255        1,197
NR-ER                       5,275      6,195            674        1,594
NR-ER-LBD                   5,931      7,058            295        1,422
NR-PPAR-gamma               5,513      6,582            167        1,236
SR-ARE                      4,952      5,784            788        1,620
SR-ATAD5                    6,006      7,163            218        1,375
SR-HSE                      5,492      6,527            316        1,351
SR-MMP                      4,945      5,777            781        1,613
SR-p53              

##  PyTorch Dataset


In [ ]:
class Tox21Dataset(Dataset):
    """
    PyTorch Dataset for Tox21 multi-task learning.

    PyTorch's DataLoader requires data in this specific format.
    __len__     → how many samples total
    __getitem__ → how to get one sample by index

    NaN values in Y are preserved as torch.nan.
    The loss function handles them — not the dataset.
    """

    def __init__(self, X, Y):
      
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        
        return self.X[idx], self.Y[idx]


train_dataset = Tox21Dataset(X_train, Y_train)
val_dataset   = Tox21Dataset(X_val,   Y_val)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f'Train dataset : {len(train_dataset):,} compounds')
print(f'Val dataset   : {len(val_dataset):,} compounds')
print(f'Train batches : {len(train_loader)} batches of {BATCH_SIZE}')
print(f'Feature dim   : {train_dataset.X.shape[1]}')
print(f'Label dim     : {train_dataset.Y.shape[1]}')


Train dataset : 6,656 compounds
Val dataset   : 1,175 compounds
Train batches : 52 batches of 128
Feature dim   : 2242
Label dim     : 12


##  Multi-Task Neural Network Architecture

```
Input (2242)  →  BatchNorm
    →  Linear 2242→1024  BatchNorm  ReLU  Dropout(0.3)   ← shared
    →  Linear 1024→512   BatchNorm  ReLU  Dropout(0.3)   ← shared
    →  Linear  512→256   BatchNorm  ReLU  Dropout(0.3)   ← shared
         ↙  ↙  ↙  ...  ↘  (12 branches)
Linear 256→64  ReLU  Dropout(0.15)  Linear 64→1  Sigmoid
```
The shared layers are updated using gradient signal from ALL 12 endpoints simultaneously.


In [ ]:
class MultiTaskTox21Net(nn.Module):
    """
    Multi-Task Neural Network for Tox21.

    Architecture:
      3 shared layers → learn general molecular toxicity patterns
                         from ALL 12 endpoints simultaneously
      12 task heads   → each specialises for one assay

    Key design choices:
    - BatchNorm before ReLU: stabilises training on sparse Morgan bits
    - Dropout(0.3): randomly silences 30% of neurons → prevents memorisation
    - Sigmoid at the end of each head → outputs are probabilities 0-1
    - nn.ModuleList: registers all 12 heads so PyTorch tracks their weights
    """

    def __init__(self, input_dim, n_tasks, dropout=0.3):
        super(MultiTaskTox21Net, self).__init__()

    
        self.input_bn = nn.BatchNorm1d(input_dim)

      
        self.shared = nn.Sequential(
            # Layer 1: 2242 → 512
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),

            # Layer 2: 512 → 256
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),

            # Layer 3: 256 → 128
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

       
        self.task_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(128, 32),
                nn.ReLU(),
                nn.Dropout(dropout / 2),  
                nn.Linear(32, 1),
                nn.Sigmoid()             
            for _ in range(n_tasks)
        ])

    def forward(self, x):
        """
        x      : tensor (batch_size, 2242)
        returns: tensor (batch_size, 12)  — 12 toxicity probabilities
        """
        x           = self.input_bn(x)
        shared_repr = self.shared(x)          # (batch, 256)

       
        outputs = torch.cat(
            [head(shared_repr) for head in self.task_heads],
            dim=1
        )   # (batch_size, 12)

        return outputs


INPUT_DIM = X_train.shape[1]     # 2242
N_TASKS   = len(toxicity_labels)  # 12

model = MultiTaskTox21Net(
    input_dim = INPUT_DIM,
    n_tasks   = N_TASKS,
    dropout   = DROPOUT_RATE,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTotal trainable parameters: {total_params:,}')
print(f'Input dimension  : {INPUT_DIM}')
print(f'Number of tasks  : {N_TASKS}')


MultiTaskTox21Net(
  (input_bn): BatchNorm1d(2242, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (shared): Sequential(
    (0): Linear(in_features=2242, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.1, inplace=False)
  )
  (task_heads): ModuleList(
    (0-11): 12 x Sequential(
      (0): Linear(in_features=128, out_features=32, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.05, inplace=False)
      (3): Linear(in_features=32, out_features=1, bia

## Masked Loss Function





In [ ]:
class MaskedBCELoss(nn.Module):
    """
    Binary Cross-Entropy Loss that ignores NaN labels.

    Standard BCE formula:
      BCE = -[ y * log(p) + (1-y) * log(1-p) ]
        y = true label (0 or 1)
        p = predicted probability (from Sigmoid)

    We add class weights to penalise missing toxic compounds more.
    We mask out NaN positions so they contribute 0 to the gradient.

    Why use BCELoss (not BCEWithLogitsLoss)?
    Because our model already applies Sigmoid() in the task heads.
    BCEWithLogitsLoss expects raw logits — it would double-apply the sigmoid.
    """

    def __init__(self, pos_weights=None):
        super(MaskedBCELoss, self).__init__()
        self.pos_weights = pos_weights   # tensor shape (n_tasks,)

    def forward(self, predictions, targets):
        """
        predictions : (batch_size, 12) — sigmoid probabilities from model
        targets     : (batch_size, 12) — true labels (0, 1, or NaN)
        """
        
        mask = ~torch.isnan(targets)

       
        targets_clean = torch.where(mask, targets, torch.zeros_like(targets))

       
        loss_per_element = nn.functional.binary_cross_entropy(
            predictions, targets_clean, reduction='none'
        )   

        
        if self.pos_weights is not None:
            weight_matrix = torch.where(
                targets_clean == 1,
                self.pos_weights.unsqueeze(0),    
                torch.ones_like(targets_clean)    
            )
            loss_per_element = loss_per_element * weight_matrix

        
        masked_loss = loss_per_element * mask.float()

        
        n_valid = mask.float().sum()
        if n_valid > 0:
            return masked_loss.sum() / n_valid
        else:
            return torch.tensor(0.0, requires_grad=True)



pw_tensor = torch.tensor(
    [class_weights.get(ep, 1.0) for ep in toxicity_labels],
    dtype=torch.float32
).to(DEVICE)

criterion = MaskedBCELoss(pos_weights=pw_tensor)

print('MaskedBCELoss defined.')
print('Class weights per endpoint (residual after SMOTE):')
for ep, w in zip(toxicity_labels, pw_tensor.cpu().numpy()):
    print(f'  {ep:<22}: {w:.2f}x')


MaskedBCELoss defined.
Class weights per endpoint (residual after SMOTE):
  NR-AR                 : 4.09x
  NR-AR-LBD             : 4.23x
  NR-AhR                : 3.02x
  NR-Aromatase          : 3.93x
  NR-ER                 : 2.89x
  NR-ER-LBD             : 3.96x
  NR-PPAR-gamma         : 4.33x
  SR-ARE                : 2.57x
  SR-ATAD5              : 4.21x
  SR-HSE                : 3.83x
  SR-MMP                : 2.58x
  SR-p53                : 3.81x


## Optimizer & Learning Rate Scheduler

In [10]:


optimizer = optim.Adam(
    model.parameters(),
    lr           = LEARNING_RATE,   # 0.001 initial LR
    weight_decay = 1e-3,            # L2 regularisation strength
)



scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max  = EPOCHS,    # full decay over all epochs
    eta_min= 1e-6,      # minimum LR
)

print(f'Optimizer  : Adam  lr={LEARNING_RATE}  weight_decay=1e-3')
print(f'Scheduler  : CosineAnnealingLR  T_max={EPOCHS}  eta_min=1e-6')


Optimizer  : Adam  lr=0.0001  weight_decay=1e-3
Scheduler  : CosineAnnealingLR  T_max=200  eta_min=1e-6


## Training Functions

One epoch = one complete pass through the training data.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """One full pass through training data. Returns average loss."""
    model.train()   # enables dropout + batchnorm running stats update
    total_loss = 0.0
    n_batches  = 0

    for X_batch, Y_batch in loader:
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        optimizer.zero_grad()              

        preds = model(X_batch)             

        loss = criterion(preds, Y_batch)   

        loss.backward()                    

       
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()                   

        total_loss += loss.item()
        n_batches  += 1

    return total_loss / n_batches


def evaluate(model, loader, criterion, device, toxicity_labels):
    """Evaluate on val set. Returns (avg_loss, auc_per_endpoint, mean_auc)."""
    model.eval()   
    total_loss = 0.0
    n_batches  = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():   
        for X_batch, Y_batch in loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)
            preds   = model(X_batch)
            loss    = criterion(preds, Y_batch)
            total_loss += loss.item()
            n_batches  += 1
            all_preds.append(preds.cpu().numpy())
            all_labels.append(Y_batch.cpu().numpy())

    all_preds  = np.vstack(all_preds)    
    all_labels = np.vstack(all_labels)   

    aucs = {}
    for i, ep in enumerate(toxicity_labels):
        mask   = ~np.isnan(all_labels[:, i])
        y_true = all_labels[mask, i]
        y_pred = all_preds[mask, i]
        if len(np.unique(y_true)) == 2:
            aucs[ep] = roc_auc_score(y_true, y_pred)
        else:
            aucs[ep] = float('nan')

    mean_auc = float(np.nanmean(list(aucs.values())))
    avg_loss = total_loss / n_batches if n_batches > 0 else 0.0
    return avg_loss, aucs, mean_auc


print('train_one_epoch() defined')
print('evaluate()        defined')


train_one_epoch() defined
evaluate()        defined


## Run Training

In [12]:
# ── Training state ────────────────────────────────────────────────
train_losses     = []
val_losses       = []
val_aucs         = []

best_val_loss    = float('inf')
best_val_auc     = 0.0
patience_counter = 0
best_model_state = None

print('=' * 65)
print('TRAINING MULTI-TASK NEURAL NETWORK')
print('=' * 65)
print(f'{"Epoch":<8} {"Train Loss":>12} {"Val Loss":>12} {"Mean AUC":>12} {"LR":>12}')
print('-' * 58)

for epoch in range(1, EPOCHS + 1):

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)

    val_loss, epoch_aucs, mean_auc = evaluate(
        model, val_loader, criterion, DEVICE, toxicity_labels
    )

    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_aucs.append(mean_auc)

    current_lr = optimizer.param_groups[0]['lr']

    if epoch % 5 == 0 or epoch == 1:
        print(f'{epoch:<8} {train_loss:>12.4f} {val_loss:>12.4f} '
              f'{mean_auc:>12.4f} {current_lr:>12.2e}')

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_val_auc     = mean_auc
        patience_counter = 0
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        torch.save(best_model_state, 'mt_models/best_model.pt')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch} '
                  f'(no improvement for {PATIENCE} epochs)')
            break


model.load_state_dict(best_model_state)
print(f'\nTraining complete!')
print(f'Best val loss : {best_val_loss:.4f}')
print(f'Best mean AUC : {best_val_auc:.4f}')
print(f'Model restored to best weights.')


TRAINING MULTI-TASK NEURAL NETWORK
Epoch      Train Loss     Val Loss     Mean AUC           LR
----------------------------------------------------------
1              0.7661       0.7307       0.6505     1.00e-04
5              0.4047       0.5023       0.7981     9.98e-05
10             0.2297       0.4831       0.7914     9.94e-05
15             0.1235       0.5230       0.7831     9.86e-05
20             0.0751       0.5664       0.7854     9.76e-05
25             0.0536       0.5988       0.7841     9.62e-05
30             0.0434       0.6206       0.7849     9.46e-05

Early stopping at epoch 32 (no improvement for 25 epochs)

Training complete!
Best val loss : 0.4797
Best mean AUC : 0.7970
Model restored to best weights.


## Training Curves

In [13]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_ran = range(1, len(train_losses) + 1)

ax1.plot(epochs_ran, train_losses, color='#378ADD', lw=2, label='Train loss')
ax1.plot(epochs_ran, val_losses,   color='#E24B4A', lw=2, label='Val loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Masked BCE Loss')
ax1.set_title('Training & Validation Loss\n'
              '(Large gap between lines = overfitting)', fontsize=11)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_ran, val_aucs, color='#639922', lw=2, label='Mean val AUC')
ax2.axhline(best_val_auc, color='black', linestyle='--', lw=1,
            label=f'Best AUC = {best_val_auc:.4f}')
ax2.axhline(0.8476, color='#EF9F27', linestyle=':', lw=1.5,
            label='XGBoost baseline 0.8476', alpha=0.8)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Mean ROC-AUC (all 12 endpoints)')
ax2.set_title('Validation AUC over Training\n'
              '(Orange dotted = XGBoost baseline to beat)', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0.5, 1.0)

plt.suptitle('Multi-Task Neural Network — Training Progress', fontsize=13)
plt.tight_layout()
plt.savefig('mt_plots/training_curves.png', dpi=150)
plt.show()
print('Saved → mt_plots/training_curves.png')


Saved → mt_plots/training_curves.png


## Final Evaluation Per Endpoint

In [14]:
model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for X_batch, Y_batch in val_loader:
        preds = model(X_batch.to(DEVICE))
        all_preds.append(preds.cpu().numpy())
        all_labels.append(Y_batch.numpy())

all_preds  = np.vstack(all_preds)    
all_labels = np.vstack(all_labels)   

# XGBoost baseline AUCs for direct comparison
XGB_BASELINE = {
    'NR-AR': 0.8060, 'NR-AR-LBD': 0.8654, 'NR-AhR': 0.9063,
    'NR-Aromatase': 0.8680, 'NR-ER': 0.7225, 'NR-ER-LBD': 0.8317,
    'NR-PPAR-gamma': 0.8442, 'SR-ARE': 0.8401, 'SR-ATAD5': 0.8817,
    'SR-HSE': 0.8087, 'SR-MMP': 0.9242, 'SR-p53': 0.8723,
}

results = []

print('FINAL RESULTS — Multi-Task Neural Network vs XGBoost Baseline')
print('=' * 75)
print(f'{"Endpoint":<22} {"N labelled":>10} {"N toxic":>8} '
      f'{"NN AUC":>8} {"XGB AUC":>9} {"Delta":>8} {"PR-AUC":>8}')
print('-' * 75)

for i, ep in enumerate(toxicity_labels):
    mask   = ~np.isnan(all_labels[:, i])
    y_true = all_labels[mask, i]
    y_pred = all_preds[mask, i]

    if len(np.unique(y_true)) == 2:
        nn_auc  = roc_auc_score(y_true, y_pred)
        pr_auc  = average_precision_score(y_true, y_pred)
    else:
        nn_auc = pr_auc = float('nan')

    xgb_auc = XGB_BASELINE.get(ep, 0)
    delta   = nn_auc - xgb_auc
    arrow   = '▲' if delta > 0 else '▼'
    n_toxic = int(y_true.sum()) if not np.isnan(y_true).any() else 0

    results.append({'endpoint': ep, 'n_val': int(mask.sum()),
                    'n_toxic': n_toxic, 'nn_auc': round(nn_auc, 4),
                    'xgb_auc': xgb_auc, 'delta': round(delta, 4),
                    'pr_auc': round(pr_auc, 4)})

    print(f'{ep:<22} {int(mask.sum()):>10,} {n_toxic:>8,} '
          f'{nn_auc:>8.4f} {xgb_auc:>9.4f} {arrow}{abs(delta):>6.4f} {pr_auc:>8.4f}')

results_df = pd.DataFrame(results)
mean_nn  = results_df['nn_auc'].mean()
mean_xgb = results_df['xgb_auc'].mean()

print('-' * 75)
print(f'{"MEAN":<22} {"":>10} {"":>8} '
      f'{mean_nn:>8.4f} {mean_xgb:>9.4f} '
      f'{"▲" if mean_nn > mean_xgb else "▼"}{abs(mean_nn-mean_xgb):>6.4f}')

print(f'\nNN mean AUC  : {mean_nn:.4f}')
print(f'XGB mean AUC : {mean_xgb:.4f}  (previous baseline)')
print(f'Improvement  : {mean_nn - mean_xgb:+.4f}')

results_df.to_csv('mt_results/final_results.csv', index=False)
print('\nSaved → mt_results/final_results.csv')

# ── ROC curves ────────────────────────────────────────────────────
COLORS = ['#378ADD','#639922','#D85A30','#534AB7','#0F6E56',
          '#BA7517','#A32D2D','#185FA5','#7F77DD','#1D9E75','#E24B4A','#EF9F27']

fig, ax = plt.subplots(figsize=(10, 8))
for i, ep in enumerate(toxicity_labels):
    mask   = ~np.isnan(all_labels[:, i])
    y_true = all_labels[mask, i]
    y_pred = all_preds[mask, i]
    if len(np.unique(y_true)) == 2:
        fpr, tpr, _ = roc_curve(y_true, y_pred)
        auc_v = results_df.loc[results_df['endpoint']==ep, 'nn_auc'].values[0]
        ax.plot(fpr, tpr, color=COLORS[i], lw=1.5,
                label=f'{ep}  ({auc_v:.3f})')

ax.plot([0,1],[0,1],'k--', lw=1, alpha=0.4, label='Random (0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves — Multi-Task Neural Network\n'
             f'Mean AUC = {mean_nn:.4f}  (XGBoost baseline = 0.8476)')
ax.legend(loc='lower right', fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('mt_plots/roc_curves.png', dpi=150)
plt.show()
print('Saved → mt_plots/roc_curves.png')


FINAL RESULTS — Multi-Task Neural Network vs XGBoost Baseline
Endpoint               N labelled  N toxic   NN AUC   XGB AUC    Delta   PR-AUC
---------------------------------------------------------------------------
NR-AR                       1,090       46   0.7337    0.8060 ▼0.0723   0.4743
NR-AR-LBD                   1,007       36   0.8660    0.8654 ▲0.0006   0.6267
NR-AhR                        961      120   0.8242    0.9063 ▼0.0821   0.5129
NR-Aromatase                  856       45   0.7776    0.8680 ▼0.0904   0.3125
NR-ER                         918      119   0.7356    0.7225 ▲0.0131   0.4917
NR-ER-LBD                   1,024       55   0.9012    0.8317 ▲0.0695   0.6031
NR-PPAR-gamma                 937       19   0.7748    0.8442 ▼0.0694   0.1441
SR-ARE                        880      154   0.7539    0.8401 ▼0.0862   0.4236
SR-ATAD5                    1,066       46   0.7671    0.8817 ▼0.1146   0.2647
SR-HSE                        975       56   0.7790    0.8087 ▼0.0297  

##  Prediction Function for New Molecules



In [15]:
def predict_toxicity(smiles_input, model, scaler, toxicity_labels, device):
    """
    Predict toxicity probabilities for one or more molecules.

    Parameters
    ----------
    smiles_input   : str or list of str — SMILES string(s)
    model          : trained MultiTaskTox21Net
    scaler         : fitted StandardScaler (saved in mt_models/scaler.pkl)
    toxicity_labels: list of 12 endpoint names
    device         : torch device

    Returns
    -------
    pandas DataFrame — rows = molecules, columns = endpoints
    Values are probabilities 0-1 (higher = more likely toxic)
    """
    if isinstance(smiles_input, str):
        smiles_input = [smiles_input]

    results = []

    for smi in smiles_input:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            print(f'Invalid SMILES: {smi}')
            results.append([np.nan] * len(toxicity_labels))
            continue

        # ── Compute the same 5 feature layers as training ─────────
        morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
        morgan = np.array(list(morgan_gen.GetFingerprint(mol)), dtype=np.float32)
        maccs  = np.array(list(MACCSkeys.GenMACCSKeys(mol)), dtype=np.float32)
        desc   = np.array(compute_descriptors(mol), dtype=np.float32)

        mw_  = desc[0]; lp_ = desc[1]; tp_ = desc[3]
        hbd_ = desc[4]; hba_= desc[5]
        eng = np.array([
            float(mw_ < 500), float(lp_ < 5),
            float(hbd_ <= 5), float(hba_ <= 10),
            float(mw_<500 and lp_<5 and hbd_<=5 and hba_<=10),
            float(4 - sum([mw_<500, lp_<5, hbd_<=5, hba_<=10])),
            lp_ * mw_, tp_ / (mw_ + 1), hbd_ + hba_, np.log1p(mw_)
        ], dtype=np.float32)
        zinc = np.array(compute_zinc(mol), dtype=np.float32)

        feat = np.concatenate([morgan, maccs, desc, eng, zinc])
        feat = np.nan_to_num(feat, nan=0.0)

        # ── Scale and predict ──────────────────────────────────────
        feat_scaled = scaler.transform(feat.reshape(1, -1)).astype(np.float32)

        model.eval()
        with torch.no_grad():
            tensor_in = torch.tensor(feat_scaled).to(device)
            probs = model(tensor_in).cpu().numpy().flatten()  # (12,)

        results.append(probs)

    pred_df = pd.DataFrame(results, columns=toxicity_labels, index=smiles_input)
    return pred_df


# ── Example predictions ───────────────────────────────────────────
test_smiles = [
    'CCOc1ccc2nc(S(N)(=O)=O)sc2c1',    # Acetazolamide (drug)
    'CC(=O)Oc1ccccc1C(=O)O',           # Aspirin (known safe)
    'c1ccc2c(c1)cc1ccc3cccc4ccc2c1c34', # Pyrene (known toxic PAH)
    'CC(C)(c1ccc(O)cc1)c1ccc(O)cc1',   # Bisphenol A (endocrine disruptor)
    'CCC(=C(c1ccccc1)c1ccccc1)c1ccc(OCCN(C)C)cc1',  # Tamoxifen (ER modulator)
]

pred_df = predict_toxicity(test_smiles, model, scaler, toxicity_labels, DEVICE)

print('TOXICITY PREDICTIONS (probability of being toxic)')
print('=' * 80)
print('  0.0 = very likely safe    |    1.0 = very likely toxic')
print('=' * 80)
for smi in test_smiles:
    print(f'\nMolecule: {smi}')
    row = pred_df.loc[smi]
    for ep in toxicity_labels:
        prob  = row[ep]
        flag  = '⚠️  TOXIC' if prob > 0.5 else '✅ safe'
        bar   = '█' * int(prob * 20)
        print(f'  {ep:<22}: {prob:.3f}  {bar:<20}  {flag}')


TOXICITY PREDICTIONS (probability of being toxic)
  0.0 = very likely safe    |    1.0 = very likely toxic

Molecule: CCOc1ccc2nc(S(N)(=O)=O)sc2c1
  NR-AR                 : 0.190  ███                   ✅ safe
  NR-AR-LBD             : 0.156  ███                   ✅ safe
  NR-AhR                : 0.857  █████████████████     ⚠️  TOXIC
  NR-Aromatase          : 0.314  ██████                ✅ safe
  NR-ER                 : 0.548  ██████████            ⚠️  TOXIC
  NR-ER-LBD             : 0.301  ██████                ✅ safe
  NR-PPAR-gamma         : 0.180  ███                   ✅ safe
  SR-ARE                : 0.644  ████████████          ⚠️  TOXIC
  SR-ATAD5              : 0.309  ██████                ✅ safe
  SR-HSE                : 0.322  ██████                ✅ safe
  SR-MMP                : 0.728  ██████████████        ⚠️  TOXIC
  SR-p53                : 0.319  ██████                ✅ safe

Molecule: CC(=O)Oc1ccccc1C(=O)O
  NR-AR                 : 0.050  █                     ✅ safe
  

##  Saving Files

In [16]:
torch.save({
    'model_state_dict' : model.state_dict(),
    'input_dim'        : INPUT_DIM,
    'n_tasks'          : N_TASKS,
    'dropout'          : DROPOUT_RATE,
    'toxicity_labels'  : toxicity_labels,
    'best_val_auc'     : best_val_auc,
    'best_val_loss'    : best_val_loss,
}, 'mt_models/final_model.pt')

with open('mt_models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

results_df.to_csv('mt_results/final_results.csv', index=False)

print('All outputs saved:')
print('  mt_models/final_model.pt      ← trained model weights + config')
print('  mt_models/best_model.pt       ← best checkpoint during training')
print('  mt_models/scaler.pkl          ← StandardScaler (required for prediction)')
print('  mt_results/final_results.csv  ← AUC scores per endpoint')
print('  mt_plots/training_curves.png')
print('  mt_plots/roc_curves.png')
print()
print('To load model later:')
print("  ckpt  = torch.load('mt_models/final_model.pt')")
print("  model = MultiTaskTox21Net(ckpt['input_dim'], ckpt['n_tasks'], ckpt['dropout'])")
print("  model.load_state_dict(ckpt['model_state_dict'])")
print()
print('=' * 55)
print('FINAL SUMMARY')
print('=' * 55)
print(f'NN  mean AUC  : {results_df["nn_auc"].mean():.4f}')
print(f'XGB mean AUC  : {results_df["xgb_auc"].mean():.4f}  (previous baseline)')
print(f'Improvement   : {results_df["nn_auc"].mean() - results_df["xgb_auc"].mean():+.4f}')
print(f'PR-AUC mean   : {results_df["pr_auc"].mean():.4f}')
best_ep  = results_df.loc[results_df['nn_auc'].idxmax()]
worst_ep = results_df.loc[results_df['nn_auc'].idxmin()]
print(f'Best endpoint : {best_ep["endpoint"]}  (AUC={best_ep["nn_auc"]:.4f})')
print(f'Worst endpoint: {worst_ep["endpoint"]}  (AUC={worst_ep["nn_auc"]:.4f})')


All outputs saved:
  mt_models/final_model.pt      ← trained model weights + config
  mt_models/best_model.pt       ← best checkpoint during training
  mt_models/scaler.pkl          ← StandardScaler (required for prediction)
  mt_results/final_results.csv  ← AUC scores per endpoint
  mt_plots/training_curves.png
  mt_plots/roc_curves.png

To load model later:
  ckpt  = torch.load('mt_models/final_model.pt')
  model = MultiTaskTox21Net(ckpt['input_dim'], ckpt['n_tasks'], ckpt['dropout'])
  model.load_state_dict(ckpt['model_state_dict'])

FINAL SUMMARY
NN  mean AUC  : 0.8076
XGB mean AUC  : 0.8476  (previous baseline)
Improvement   : -0.0400
PR-AUC mean   : 0.4256
Best endpoint : NR-AR-LBD  (AUC=0.8987)
Worst endpoint: NR-ER  (AUC=0.7467)
